# Blok E — Kondisionalitas / generalisasi

Apakah temuan faktorial (sel4 terbaik, interaksi attention×P menguntungkan) bertahan di
luar kondisi baku (trust 0,5 · gamma 0,99 · beban 4× · gamma_est_wait 0,0559)?

Tujuh kondisi, masing-masing faktorial 2×2 penuh. **Satu faktor berubah per kondisi.**

> ⚠ **Dua jenis pertanyaan yang TIDAK boleh digabung dalam satu klaim:**
>
> | kondisi | jenis faktor | menguji |
> |---|---|---|
> | trust awal 0,3 / 0,7 | **lingkungan** | validitas eksternal |
> | beban 6× | **lingkungan/substrat** | validitas eksternal |
> | `gamma_est_wait` ×0,5 / ×2 | **lingkungan/perilaku pengguna** | validitas eksternal |
> | gamma 0,95 / 0,999 | **hiperparameter algoritma** | sensitivitas penyetelan |
>
> Baris terakhir adalah diskon PPO/GAE, bukan properti lingkungan. Efeknya juga
> **teredam** oleh `max_step_gap=4` yang memutus rantai *bootstrap* antar-transisi
> berjauhan waktu — hasil null di sana wajar dan tetap layak dilaporkan.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import _analisis_bab5 as A
pd.set_option('display.width', 235); pd.set_option('display.max_columns', 60)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True, 'grid.alpha': .3})
HORIZON = '90d'
B  = 'master_hybrid_ppo_dgr_90d_cwtfail120pen-2_preffeat_pairout'
B6 = 'master_hybrid_ppo_dgr_90d_load6x_cwtfail120pen-2_preffeat_pairout'
A.ARAH_BAIK.update({'gini_UTIL': 'min', 'gini_ANTRE': 'min'})
SUF = {'sel1': '_noattn_pure3', 'sel2': '_pure3',
       'sel3': '_pg0.1_noattn_pure3', 'sel4': '_pg0.1_pure3'}

def nilai(tag, m):
    if m == 'gini_UTIL':  return A.gini_stasiun(tag, 'gini_utilisasi', HORIZON)
    if m == 'gini_ANTRE': return A.gini_stasiun(tag, 'gini_antrean',   HORIZON)
    return A.unit_stat(tag, m, HORIZON)

KOND = [('baku',            B,  ''),
        ('trust 0,3',       B,  '_it0.3'),
        ('trust 0,7',       B,  '_it0.7'),
        ('gamma 0,95',      B,  '_g0.95'),
        ('gamma 0,999',     B,  '_g0.999'),
        ('gw x0,5',         B,  '_gw0.0279513'),
        ('gw x2',           B,  '_gw0.111805'),
        ('beban 6x',        B6, '')]

def tag(basis, sel, kond):
    return f'{basis}{SUF[sel]}{kond}'

for nm, basis, k in KOND:
    n = [len(nilai(tag(basis, s, k), 'acc')[2]) for s in SUF]
    print(f'{nm:12s} n checkpoint per sel = {n}')

baku         n checkpoint per sel = [3, 3, 3, 3]
trust 0,3    n checkpoint per sel = [3, 3, 3, 3]
trust 0,7    n checkpoint per sel = [5, 5, 5, 5]
gamma 0,95   n checkpoint per sel = [3, 3, 3, 3]
gamma 0,999  n checkpoint per sel = [3, 3, 3, 3]
gw x0,5      n checkpoint per sel = [3, 3, 3, 3]
gw x2        n checkpoint per sel = [3, 3, 3, 3]


beban 6x     n checkpoint per sel = [3, 3, 3, 3]


## E.1 — Metrik inti lintas kondisi

In [2]:
M = ['gini_UTIL', 'wait', 'acc', 'trust', 'rec_entropy', 'herding']
baris = []
for nm, basis, k in KOND:
    for sel in SUF:
        r = {'kondisi': nm, 'sel': sel}
        for m in M:
            r[m] = nilai(tag(basis, sel, k), m)[0]
        baris.append(r)
kon = pd.DataFrame(baris).set_index(['kondisi', 'sel'])
display(kon.round(4))

gini_UTIL      wait     acc   trust  rec_entropy  herding
kondisi     sel                                                            
baku        sel1     0.0289   95.9557  0.6700  0.3666       0.0602   0.3658
            sel2     0.0393   89.6613  0.7021  0.3935       0.0788   0.3471
            sel3     0.0521   84.1083  0.7034  0.3960       0.0747   0.3499
            sel4     0.0206   66.2480  0.7194  0.4348       0.1182   0.3063
trust 0,3   sel1     0.0391   76.8760  0.6722  0.3080       0.0901   0.3343
            sel2     0.0576   86.8043  0.6644  0.2992       0.0883   0.3387
            sel3     0.0593   85.2598  0.6652  0.2997       0.0784   0.3477
            sel4     0.0295   76.3345  0.6674  0.3118       0.0954   0.3303
trust 0,7   sel1     0.0660   76.8297  0.7525  0.4906       0.0905   0.3343
            sel2     0.0560   71.0091  0.7720  0.5087       0.1027   0.3200
            sel3     0.0448   73.7102  0.7496  0.4958       0.0879   0.3370
            sel4     0.0504   68.9026  0.7643  0.5135       0.1065   0.3166
gamma 0,95  sel1     0.0247   69.0491  0.7173  0.4216       0.1020   0.3236
            sel2     0.0367   67.4993  0.7196  0.4283       0.1165   0.3050
            sel3     0.0298   66.2030  0.7282  0.4316       0.0980   0.3279
            sel4     0.0488   70.1003  0.7155  0.4209       0.1087   0.3139
gamma 0,999 sel1     0.0955   87.0472  0.7378  0.3975       0.0874   0.3352
            sel2     0.0481   67.7524  0.7227  0.4257       0.1047   0.3214
            sel3     0.1171  151.7522  0.7074  0.3749       0.0620   0.3598
            sel4     0.0575   79.6247  0.7121  0.3981       0.0727   0.3515
gw x0,5     sel1     0.0277   68.9195  0.7258  0.4275       0.0903   0.3342
            sel2     0.0324   73.7642  0.7105  0.4127       0.1194   0.2984
            sel3     0.0449   77.2984  0.7080  0.4059       0.0924   0.3343
            sel4     0.0453   72.1390  0.7175  0.4281       0.0956   0.3293
gw x2       sel1     0.1275  205.3865  0.7458  0.3585       0.0530   0.3675
            sel2     0.0441   67.5719  0.7155  0.4297       0.1056   0.3194
            sel3     0.0411   70.5486  0.7139  0.4171       0.0943   0.3289
            sel4     0.0337   65.6245  0.7326  0.4294       0.1048   0.3179
beban 6x    sel1     0.0824  136.9264  0.6564  0.3409       0.0404   0.3821
            sel2     0.1178  251.5948  0.7099  0.3582       0.0589   0.3623
            sel3     0.1768  223.3252  0.7121  0.3217       0.0393   0.3828
            sel4     0.0399   85.5575  0.6829  0.3667       0.0820   0.3414

In [3]:
print('PERINGKAT sel4 pada tiap kondisi (1 = terbaik dari 4 sel)\n')
baris = []
for nm, basis, k in KOND:
    r = {'kondisi': nm}
    for m in M:
        v = {s: nilai(tag(basis, s, k), m)[0] for s in SUF}
        arah = A.ARAH_BAIK.get(m, 'max')
        urut = sorted(v, key=lambda s: v[s] if arah == 'min' else -v[s])
        r[m] = urut.index('sel4') + 1
    baris.append(r)
pk = pd.DataFrame(baris).set_index('kondisi')
display(pk)
print(f"sel4 peringkat-1 pada {(pk == 1).sum().sum()} dari {pk.size} sel-metrik")
print('\nKondisi di mana sel4 BUKAN peringkat-1 pada gini_UTIL:')
for nm in pk.index:
    if pk.loc[nm, 'gini_UTIL'] != 1:
        print(f'   {nm}  (peringkat {pk.loc[nm, "gini_UTIL"]})')

PERINGKAT sel4 pada tiap kondisi (1 = terbaik dari 4 sel)



,gini_UTIL,wait,acc,trust,rec_entropy,herding
kondisi,,,,,,
baku,1,1,1,1,1,1
"trust 0,3",1,1,2,1,1,1
"trust 0,7",2,1,2,1,1,1
"gamma 0,95",4,4,4,4,2,2
"gamma 0,999",2,2,3,2,3,3
"gw x0,5",4,2,2,1,2,2
gw x2,1,1,2,2,2,1
beban 6x,1,1,3,1,1,1


sel4 peringkat-1 pada 24 dari 48 sel-metrik

Kondisi di mana sel4 BUKAN peringkat-1 pada gini_UTIL:
   trust 0,7  (peringkat 2)
   gamma 0,95  (peringkat 4)
   gamma 0,999  (peringkat 2)
   gw x0,5  (peringkat 4)


## E.2 — Interaksi attention × P lintas kondisi

Pertanyaan inti Blok E: apakah komplementaritas yang ditemukan di kondisi baku bertahan?

In [4]:
baris = []
for nm, basis, k in KOND:
    for m in ['gini_UTIL', 'wait', 'acc', 'rec_entropy']:
        v = {s: nilai(tag(basis, s, k), m)[0] for s in SUF}
        s1, s2, s3, s4 = v['sel1'], v['sel2'], v['sel3'], v['sel4']
        it = (s4 - s3) - (s2 - s1)
        ea = (s2 + s4) / 2 - (s1 + s3) / 2
        ep = (s3 + s4) / 2 - (s1 + s2) / 2
        arah = A.ARAH_BAIK.get(m, 'max')
        baik = (it < 0) == (arah == 'min')
        baris.append({'kondisi': nm, 'metrik': m, 'efek_attn': ea, 'efek_P': ep,
                      'interaksi': it, 'menguntungkan': baik})
inter = pd.DataFrame(baris)
display(inter.pivot(index='kondisi', columns='metrik',
                    values='interaksi').round(4))
print('\nApakah interaksi MENGUNTUNGKAN? (True = ya)')
display(inter.pivot(index='kondisi', columns='metrik', values='menguntungkan'))

metrik,acc,gini_UTIL,rec_entropy,wait
kondisi,,,,
baku,-0.0160,-0.0419,0.0250,-11.5660
beban 6x,-0.0827,-0.1722,0.0241,-252.4361
"gamma 0,95",-0.0150,0.0071,-0.0038,5.4472
"gamma 0,999",0.0197,-0.0121,-0.0065,-52.8327
"gw x0,5",0.0248,-0.0043,-0.0258,-10.0041
gw x2,0.0491,0.0761,-0.0422,132.8905
"trust 0,3",0.0100,-0.0484,0.0188,-18.8536
"trust 0,7",-0.0048,0.0156,0.0063,1.0130



Apakah interaksi MENGUNTUNGKAN? (True = ya)


metrik,acc,gini_UTIL,rec_entropy,wait
kondisi,,,,
baku,False,True,True,True
beban 6x,False,True,True,True
"gamma 0,95",False,False,False,False
"gamma 0,999",True,True,False,True
"gw x0,5",True,True,False,True
gw x2,True,False,False,False
"trust 0,3",True,True,True,True
"trust 0,7",False,False,True,False


In [5]:
g = inter[inter['metrik'] == 'gini_UTIL'].set_index('kondisi')
print('INTERAKSI pada gini_UTIL (metrik inti TK1)\n')
for nm in g.index:
    r = g.loc[nm]
    print(f"   {nm:12s} interaksi={r['interaksi']:+.5f}  "
          f"efek_attn={r['efek_attn']:+.5f}  efek_P={r['efek_P']:+.5f}  "
          f"{'menguntungkan' if r['menguntungkan'] else 'MERUGIKAN'}")
n_baik = int(g['menguntungkan'].sum())
print(f'\n>> menguntungkan pada {n_baik}/{len(g)} kondisi')
print('   Klaim yang dapat dipertanggungjawabkan hanya sekuat proporsi ini.')

INTERAKSI pada gini_UTIL (metrik inti TK1)

   baku         interaksi=-0.04188  efek_attn=-0.01058  efek_P=+0.00229  menguntungkan
   trust 0,3    interaksi=-0.04836  efek_attn=-0.00569  efek_P=-0.00392  menguntungkan
   trust 0,7    interaksi=+0.01558  efek_attn=-0.00224  efek_P=-0.01342  MERUGIKAN
   gamma 0,95   interaksi=+0.00708  efek_attn=+0.01549  efek_P=+0.00859  MERUGIKAN
   gamma 0,999  interaksi=-0.01212  efek_attn=-0.05351  efek_P=+0.01548  menguntungkan
   gw x0,5      interaksi=-0.00431  efek_attn=+0.00258  efek_P=+0.01510  menguntungkan
   gw x2        interaksi=+0.07607  efek_attn=-0.04536  efek_P=-0.04842  MERUGIKAN
   beban 6x     interaksi=-0.17223  efek_attn=-0.05073  efek_P=+0.00820  menguntungkan

>> menguntungkan pada 5/8 kondisi
   Klaim yang dapat dipertanggungjawabkan hanya sekuat proporsi ini.


## E.3 — Stabilitas antar-checkpoint lintas kondisi

Kondisi yang menaikkan variansi antar-checkpoint menandakan pelatihan menjadi rapuh —
sekalipun reratanya tampak baik.

In [6]:
baris = []
for nm, basis, k in KOND:
    for sel in SUF:
        v = nilai(tag(basis, sel, k), 'gini_UTIL')
        baris.append({'kondisi': nm, 'sel': sel, 'mean': v[0], 'sd': v[1],
                      'cv': v[1] / v[0] if v[0] else np.nan,
                      'rentang': f'[{v[2].min():.4f}, {v[2].max():.4f}]',
                      'n': len(v[2])})
st = pd.DataFrame(baris)
display(st.pivot(index='kondisi', columns='sel', values='cv').round(3))
print('CV = SD/mean pada gini_UTIL. Nilai besar = pelatihan tak stabil di kondisi itu.\n')
tinggi = st[st['cv'] > 0.5].sort_values('cv', ascending=False)
print('Sel-kondisi dgn CV > 0,5 (sangat tak stabil):')
for _, r in tinggi.iterrows():
    print(f"   {r['kondisi']:12s} {r['sel']}  cv={r['cv']:.2f}  rentang={r['rentang']}")

sel,sel1,sel2,sel3,sel4
kondisi,,,,
baku,0.326,0.320,0.235,0.229
beban 6x,0.385,0.879,0.280,0.074
"gamma 0,95",0.441,0.255,0.179,0.199
"gamma 0,999",0.679,0.044,0.651,0.263
"gw x0,5",0.149,0.164,0.336,0.735
gw x2,0.689,0.200,0.285,0.529
"trust 0,3",0.369,0.577,0.741,0.416
"trust 0,7",0.443,1.025,0.326,0.293


CV = SD/mean pada gini_UTIL. Nilai besar = pelatihan tak stabil di kondisi itu.

Sel-kondisi dgn CV > 0,5 (sangat tak stabil):
   trust 0,7    sel2  cv=1.02  rentang=[0.0216, 0.1707]
   beban 6x     sel2  cv=0.88  rentang=[0.0239, 0.2622]
   trust 0,3    sel3  cv=0.74  rentang=[0.0256, 0.1215]
   gw x0,5      sel4  cv=0.74  rentang=[0.0154, 0.0918]
   gw x2        sel1  cv=0.69  rentang=[0.0267, 0.2408]
   gamma 0,999  sel1  cv=0.68  rentang=[0.0287, 0.1834]
   gamma 0,999  sel3  cv=0.65  rentang=[0.0385, 0.2202]
   trust 0,3    sel2  cv=0.58  rentang=[0.0319, 0.1044]
   gw x2        sel4  cv=0.53  rentang=[0.0164, 0.0583]


## E.4 — Acuan greedy pada tiap kondisi

Blok E semula membandingkan empat sel PURE3 satu sama lain saja. Tanpa pembanding
non-RL, "arsitektur ini lebih tahan terhadap kondisi X" tidak dapat dipisahkan dari
"kondisi X memang lebih sulit bagi agen apa pun". Greedy tidak perlu dilatih ulang,
sehingga acuan itu dapat dilengkapi lewat evaluasi ulang saja
(`_uji_greedy_setara_metrik.py <seed> 90d 3 <kondisi>`, 10 run tiap kondisi, ketiga
penyetaraan Blok A tetap berlaku).

Dua kondisi TIDAK memiliki padanan greedy: `gamma 0,95` dan `gamma 0,999` adalah faktor
diskon PPO/GAE — parameter **algoritma**, bukan lingkungan. Greedy tak punya fungsi
nilai sehingga perilakunya tak berubah; acuannya sengaja dipetakan ke `baku`, dan
keidentikan itu sendiri adalah informasinya: setiap perbedaan di sana murni milik agen RL.

> ### ⚠ `beban 6×` — lengan RL tidak diuji pada beban 6×
>
> `eval_pure3_beban6x_metrik.sh` memanggil `_uji_master_pure_hybrid_ppo_metrik.py`
> dengan `TAG=90d`, dan baris 57 skrip itu mengunci `K.DS` ke dataset **4×**. Berbeda
> dari `_it` dan `_gw` yang diturunkan dari tag, `_load6x` tidak memiliki penurunan
> dataset sama sekali. Terverifikasi lewat `served`: 16.392 pada lengan 6× sedangkan
> dataset 6× berisi 25.106 permintaan.
>
> Jadi kondisi itu sesungguhnya **uji transfer** — dilatih pada 6×, dievaluasi pada 4× —
> bukan uji beban berat. Karena itu acuan greedy untuk baris `beban 6×` adalah greedy
> **4×**: itulah yang setara lingkungannya. Yang diukur baris itu adalah *ketahanan
> transfer*, dan di situ hasilnya justru tajam — lihat sel di bawah.
>
> Acuan 6× yang sebenarnya (`greedy_setara_k3_load6x`) dilaporkan TERPISAH sebagai ukuran
> beratnya rezim itu sendiri; sisi RL-nya belum ada dan menunggu
> `eval_pure3_beban6x_sejati.sh` (TAG `90d6x`).

In [7]:
# Peta kondisi -> tag greedy setara lingkungan (definisi ada di _analisis_bab5.py)
def kond_token(basis, k):
    return '_load6x' if basis == B6 else k

GRD = {}
for nm, basis, k in KOND:
    GRD[nm] = A.greedy_untuk(kond_token(basis, k))

for nm, basis, k in KOND:
    t = kond_token(basis, k)
    cat = A.GREEDY_KONDISI_CATATAN.get(t, '')
    print(f"{nm:12s} -> {GRD[nm]:32s} {cat}")

baku         -> greedy_setara_k3                 
trust 0,3    -> greedy_setara_k3_it0.3           
trust 0,7    -> greedy_setara_k3_it0.7           
gamma 0,95   -> greedy_setara_k3                 gamma = parameter algoritma; acuan greedy identik dgn baku
gamma 0,999  -> greedy_setara_k3                 gamma = parameter algoritma; acuan greedy identik dgn baku
gw x0,5      -> greedy_setara_k3_gw0.0279513     
gw x2        -> greedy_setara_k3_gw0.111805      
beban 6x     -> greedy_setara_k3                 RL dilatih 6x tapi DIEVALUASI 4x -> ini uji TRANSFER; acuan greedy sengaja 4x agar setara. Acuan 6x sejati: greedy_setara_k3_load6x


In [8]:
# Metrik inti greedy_queue pada tiap kondisi.
# Hanya greedy_queue: greedy_util menilai stasiun dgn okupansi -- besaran yang SAMA
# dengan gini_UTIL -- sehingga sirkular sebagai acuan TK1 (AUDIT_METRIK.md §4).
def nilai_g(tag_g, m):
    if m == 'gini_UTIL':  return A.gini_stasiun(tag_g, 'gini_utilisasi', HORIZON, 'greedy_queue')
    if m == 'gini_ANTRE': return A.gini_stasiun(tag_g, 'gini_antrean',   HORIZON, 'greedy_queue')
    return A.unit_stat(tag_g, m, HORIZON, 'greedy_queue')

baris = []
for nm, basis, k in KOND:
    r = {'kondisi': nm}
    for m in M:
        r[m] = nilai_g(GRD[nm], m)[0]
    baris.append(r)
grd = pd.DataFrame(baris).set_index('kondisi')
print('GREEDY_QUEUE lintas kondisi (unit = run, n=10)'); print()
print(grd.round(4))

GREEDY_QUEUE lintas kondisi (unit = run, n=10)

             gini_UTIL     wait     acc   trust  rec_entropy  herding
kondisi                                                              
baku            0.1163  39.0510  0.7165  0.4366          0.0   0.4255
trust 0,3       0.1142  43.4859  0.6778  0.3371          0.0   0.4260
trust 0,7       0.1170  38.5587  0.7459  0.5113          0.0   0.4259
gamma 0,95      0.1163  39.0510  0.7165  0.4366          0.0   0.4255
gamma 0,999     0.1163  39.0510  0.7165  0.4366          0.0   0.4255
gw x0,5         0.1187  40.6587  0.7220  0.4420          0.0   0.4260
gw x2           0.1153  39.4610  0.7144  0.4305          0.0   0.4256
beban 6x        0.1163  39.0510  0.7165  0.4366          0.0   0.4255


In [9]:
# sel4 vs acuan greedy pada tiap kondisi -- selisih bertanda arah "lebih baik".
# Positif = sel4 unggul.
baris = []
for nm, basis, k in KOND:
    r = {'kondisi': nm}
    for m in M:
        v4 = nilai(tag(basis, 'sel4', k), m)[0]
        vg = nilai_g(GRD[nm], m)[0]
        tanda = -1.0 if A.ARAH_BAIK.get(m) == 'min' else 1.0
        r[m] = tanda * (v4 - vg)
    baris.append(r)
sel = pd.DataFrame(baris).set_index('kondisi')
print('sel4 MINUS greedy_queue, bertanda arah baik (+ = sel4 unggul)'); print()
print(sel.round(4))
print()
menang = (sel > 0).sum(axis=1)
for nm in sel.index:
    tag_k = kond_token(*[(b, k) for n, b, k in KOND if n == nm][0])
    catat = ' [acuan = baku]' if tag_k in ('_g0.95', '_g0.999') else ''
    catat = ' [TAK SEBANDING]' if tag_k == '_load6x' else catat
    print(f"   {nm:12s} sel4 unggul pada {menang[nm]}/{len(M)} metrik inti{catat}")

sel4 MINUS greedy_queue, bertanda arah baik (+ = sel4 unggul)

             gini_UTIL     wait     acc   trust  rec_entropy  herding
kondisi                                                              
baku            0.0957 -27.1970  0.0029 -0.0018       0.1182   0.1192
trust 0,3       0.0847 -32.8486 -0.0104 -0.0253       0.0954   0.0957
trust 0,7       0.0666 -30.3439  0.0183  0.0022       0.1065   0.1093
gamma 0,95      0.0675 -31.0494 -0.0010 -0.0156       0.1087   0.1116
gamma 0,999     0.0588 -40.5737 -0.0045 -0.0385       0.0727   0.0740
gw x0,5         0.0733 -31.4803 -0.0045 -0.0140       0.0956   0.0966
gw x2           0.0816 -26.1635  0.0182 -0.0011       0.1048   0.1077
beban 6x        0.0764 -46.5065 -0.0337 -0.0699       0.0820   0.0842

   baku         sel4 unggul pada 4/6 metrik inti
   trust 0,3    sel4 unggul pada 3/6 metrik inti
   trust 0,7    sel4 unggul pada 5/6 metrik inti
   gamma 0,95   sel4 unggul pada 3/6 metrik inti [acuan = baku]
   gamma 0,999  sel4 ungg

In [10]:
# Apakah kondisi yang menyulitkan RL juga menyulitkan greedy?
# Bila ya, kesulitan itu milik LINGKUNGAN; bila tidak, milik ARSITEKTUR.
# gamma dikeluarkan (acuan greedy-nya identik dgn baku, jadi selisihnya trivial nol);
# beban 6x dikeluarkan (lengan RL diuji di dataset berbeda).
banding = [(nm, basis, k) for nm, basis, k in KOND
           if kond_token(basis, k) not in ('_g0.95', '_g0.999', '_load6x')]
b0 = nilai_g(A.greedy_untuk(''), 'gini_UTIL')[0]
s0 = nilai(tag(B, 'sel4', ''), 'gini_UTIL')[0]

print('PERUBAHAN gini_UTIL relatif thd kondisi baku (+ = memburuk)'); print()
print(f"{'kondisi':14s} {'greedy':>10s} {'sel4':>10s}   tafsir")
for nm, basis, k in banding:
    if nm == 'baku':
        continue
    dg = nilai_g(GRD[nm], 'gini_UTIL')[0] - b0
    ds = nilai(tag(basis, 'sel4', k), 'gini_UTIL')[0] - s0
    if dg > 0 and ds > 0:
        taf = 'lingkungan lebih sulit bagi keduanya'
    elif dg <= 0 < ds:
        taf = 'kesulitan KHAS arsitektur RL'
    elif ds <= 0 < dg:
        taf = 'RL justru diuntungkan; greedy tidak'
    else:
        taf = 'keduanya membaik'
    print(f"{nm:14s} {dg:+10.4f} {ds:+10.4f}   {taf}")

PERUBAHAN gini_UTIL relatif thd kondisi baku (+ = memburuk)

kondisi            greedy       sel4   tafsir
trust 0,3         -0.0021    +0.0088   kesulitan KHAS arsitektur RL
trust 0,7         +0.0007    +0.0298   lingkungan lebih sulit bagi keduanya
gw x0,5           +0.0023    +0.0247   lingkungan lebih sulit bagi keduanya
gw x2             -0.0010    +0.0131   kesulitan KHAS arsitektur RL


### E.4b — Ketahanan transfer 6× → 4×, dan beratnya rezim 6× itu sendiri

In [11]:
# Semua baris DIEVALUASI pada 4x -> perbandingan sah.
baris = []
baris.append({'lengan': 'greedy_queue', **{m: nilai_g(A.greedy_untuk(''), m)[0] for m in M}})
for s in SUF:
    baris.append({'lengan': f'{s} dilatih-4x', **{m: nilai(tag(B, s, ''), m)[0] for m in M}})
for s in SUF:
    baris.append({'lengan': f'{s} dilatih-6x', **{m: nilai(tag(B6, s, ''), m)[0] for m in M}})
print('SEMUA dievaluasi pada dataset 4x (n=10 run / 3 checkpoint)'); print()
print(pd.DataFrame(baris).set_index('lengan').round(4))
print()
print('Degradasi gini_UTIL akibat dilatih di rezim yang salah (dilatih-6x minus dilatih-4x):')
for s in SUF:
    d = nilai(tag(B6, s, ''), 'gini_UTIL')[0] - nilai(tag(B, s, ''), 'gini_UTIL')[0]
    print(f'   {s}  {d:+.4f}')

SEMUA dievaluasi pada dataset 4x (n=10 run / 3 checkpoint)

                 gini_UTIL      wait     acc   trust  rec_entropy  herding
lengan                                                                    
greedy_queue        0.1163   39.0510  0.7165  0.4366       0.0000   0.4255
sel1 dilatih-4x     0.0289   95.9557  0.6700  0.3666       0.0602   0.3658
sel2 dilatih-4x     0.0393   89.6613  0.7021  0.3935       0.0788   0.3471
sel3 dilatih-4x     0.0521   84.1083  0.7034  0.3960       0.0747   0.3499
sel4 dilatih-4x     0.0206   66.2480  0.7194  0.4348       0.1182   0.3063
sel1 dilatih-6x     0.0824  136.9264  0.6564  0.3409       0.0404   0.3821
sel2 dilatih-6x     0.1178  251.5948  0.7099  0.3582       0.0589   0.3623
sel3 dilatih-6x     0.1768  223.3252  0.7121  0.3217       0.0393   0.3828
sel4 dilatih-6x     0.0399   85.5575  0.6829  0.3667       0.0820   0.3414

Degradasi gini_UTIL akibat dilatih di rezim yang salah (dilatih-6x minus dilatih-4x):
   sel1  +0.0535
   sel2  +0

In [12]:
# Rezim 6x SEJATI -- baru greedy yang tersedia (checkpoint RL menunggu
# eval_pure3_beban6x_sejati.sh dgn TAG '90d6x').
baris = []
for nm, t in [('greedy_queue 4x', 'greedy_setara_k3'),
              ('greedy_queue 6x', 'greedy_setara_k3_load6x')]:
    baris.append({'lengan': nm, **{m: nilai_g(t, m)[0] for m in M}})
print('Rezim 6x SEJATI'); print()
print(pd.DataFrame(baris).set_index('lengan').round(4))
print()
print('CATATAN: gini_UTIL MEMBAIK (0,1163 -> 0,0336) sementara wait melonjak 4,6x.')
print('Pada beban 6x seluruh stasiun jenuh sehingga utilisasi merata SECARA MEKANIS --')
print('tujuan pemerataan menjadi trivial di rezim itu. Alasan tersendiri untuk tidak')
print('menjadikan 6x sebagai rezim pelaporan utama.')

Rezim 6x SEJATI

                 gini_UTIL      wait     acc   trust  rec_entropy  herding
lengan                                                                    
greedy_queue 4x     0.1163   39.0510  0.7165  0.4366          0.0   0.4255
greedy_queue 6x     0.0336  178.0788  0.6457  0.3137          0.0   0.5813

CATATAN: gini_UTIL MEMBAIK (0,1163 -> 0,0336) sementara wait melonjak 4,6x.
Pada beban 6x seluruh stasiun jenuh sehingga utilisasi merata SECARA MEKANIS --
tujuan pemerataan menjadi trivial di rezim itu. Alasan tersendiri untuk tidak
menjadikan 6x sebagai rezim pelaporan utama.


### E.4c — Rezim beban 6× yang sebenarnya

Dievaluasi lewat `eval_pure3_beban6x_sejati.sh` (TAG `90d6x`), 10 run tiap sel,
checkpoint yang sama dengan lengan transfer di E.4b. Terverifikasi berjalan pada dataset
6×: `served` 19.608–22.726 (dataset berisi 25.106 permintaan; dataset 4× hanya 16.718).

Ini melengkapi sisi yang hilang — dan hasilnya **membalik pembacaan `gini_UTIL` di rezim
ini**.

In [13]:
H6 = '90d6x'
def nilai6(tag, m):
    if m == 'gini_UTIL':  return A.gini_stasiun(tag, 'gini_utilisasi', H6)
    if m == 'gini_ANTRE': return A.gini_stasiun(tag, 'gini_antrean',   H6)
    return A.unit_stat(tag, m, H6)

M6 = ['gini_UTIL', 'gini_ANTRE', 'wait', 'w_p50', 'acc', 'trust', 'served', 'herding']
baris = [{'lengan': 'greedy_queue', **{m: nilai_g('greedy_setara_k3_load6x', m)[0]
                                       for m in M6}}]
for s in SUF:
    baris.append({'lengan': s, **{m: nilai6(tag(B6, s, ''), m)[0] for m in M6}})
print('REZIM BEBAN 6x SEJATI (n=10 run / 3 checkpoint)'); print()
print(pd.DataFrame(baris).set_index('lengan').round(4))

REZIM BEBAN 6x SEJATI (n=10 run / 3 checkpoint)

              gini_UTIL  gini_ANTRE      wait      w_p50     acc   trust      served  herding
lengan                                                                                       
greedy_queue     0.0336      0.1089  178.0788   137.6974  0.6457  0.3137  24994.8000   0.5813
sel1             0.0057      0.1517  885.1050   961.1272  0.5733  0.2780  22579.4167   0.5045
sel2             0.0356      0.2525  849.1507   924.1971  0.6003  0.2978  21885.1111   0.5529
sel3             0.1304      0.3108  975.5449  1054.9160  0.6819  0.3421  19607.6667   0.5512
sel4             0.0020      0.1571  948.9733  1026.5609  0.6126  0.2704  22725.8056   0.3860


In [14]:
# gini_UTIL sel4 = 0,0020 -- nyaris nol. Sebelum dibaca sbg keberhasilan pemerataan,
# periksa TINGKAT utilisasinya: merata pada 0,3 dan merata pada 0,99 adalah dua hasil
# yang sangat berbeda meski Gini-nya sama-sama nol.
def util_per_stasiun(tag, h, ln=None):
    d = A.muat(tag, h); k = A._pilih_lengan(d, ln)
    return [[v['util_mean'] for v in r['_stasiun'].values()] for r in d['per_seed'][k]]

print(f"{'lengan':14s}{'util rata':>11s}   per-stasiun (run 0)")
u = util_per_stasiun('greedy_setara_k3_load6x', '90d', 'greedy_queue')
print(f"{'greedy_queue':14s}{np.mean(u):11.3f}   {[round(x, 2) for x in u[0]]}")
for s in SUF:
    u = util_per_stasiun(tag(B6, s, ''), H6)
    print(f"{s:14s}{np.mean(u):11.3f}   {[round(x, 2) for x in u[0]]}")
print()
print('Seluruh stasiun lengan RL terpaku ~0,99: utilisasi merata karena SEMUA jenuh,')
print('bukan karena beban dialokasikan dengan baik. Gini mendekati nol di sini adalah')
print('kesetaraan DEGENERAT -- greedy justru lebih rendah utilisasinya (0,897) sambil')
print('melayani LEBIH BANYAK EV dgn tunggu 5x lebih pendek.')

lengan          util rata   per-stasiun (run 0)
greedy_queue        0.897   [0.86, 0.94, 0.97, 0.94, 0.85, 0.81]
sel1                0.985   [1.0, 1.0, 0.98, 0.96, 0.98, 0.98]
sel2                0.948   [1.0, 1.0, 0.09, 0.0, 0.98, 1.0]
sel3                0.875   [0.99, 0.99, 0.99, 0.99, 1.0, 0.99]
sel4                0.992   [0.99, 0.99, 0.99, 0.99, 0.99, 0.99]

Seluruh stasiun lengan RL terpaku ~0,99: utilisasi merata karena SEMUA jenuh,
bukan karena beban dialokasikan dengan baik. Gini mendekati nol di sini adalah
kesetaraan DEGENERAT -- greedy justru lebih rendah utilisasinya (0,897) sambil
melayani LEBIH BANYAK EV dgn tunggu 5x lebih pendek.


In [15]:
# Ukuran yang tidak dapat dipalsukan oleh kejenuhan: throughput dan nilai mengikuti
# rekomendasi (wtolak_mean - wpatuh_mean; positif = mengikuti menguntungkan).
print(f"{'lengan':14s}{'served':>11s}{'wpatuh':>11s}{'wtolak':>11s}{'nilai ikut':>12s}")
g = {m: nilai_g('greedy_setara_k3_load6x', m)[0]
     for m in ['served', 'wpatuh_mean', 'wtolak_mean']}
print(f"{'greedy_queue':14s}{g['served']:11.0f}{g['wpatuh_mean']:11.1f}"
      f"{g['wtolak_mean']:11.1f}{g['wtolak_mean'] - g['wpatuh_mean']:12.1f}")
for s in SUF:
    r = {m: nilai6(tag(B6, s, ''), m)[0]
         for m in ['served', 'wpatuh_mean', 'wtolak_mean']}
    print(f"{s:14s}{r['served']:11.0f}{r['wpatuh_mean']:11.1f}"
          f"{r['wtolak_mean']:11.1f}{r['wtolak_mean'] - r['wpatuh_mean']:12.1f}")
print()
print('NILAI MENGIKUTI REKOMENDASI NEGATIF pada KEEMPAT sel. Di rezim 6x, mengikuti')
print('rekomendasi agen MERUGIKAN pengguna -- sementara pada greedy tetap +110,7 mnt.')
print('Ini kebalikan penuh dari temuan di rezim 4x (sel4 +54,3 mnt, tertinggi dari')
print('seluruh lengan). Batas keberlakuan artefak, dan wajib dilaporkan.')

lengan             served     wpatuh     wtolak  nilai ikut
greedy_queue        24995      138.9      249.5       110.7
sel1                22579      926.7      833.8       -92.9
sel2                21885      924.8      730.1      -194.7
sel3                19608     1024.4      761.7      -262.7


sel4                22726      952.7      942.4       -10.2



NILAI MENGIKUTI REKOMENDASI NEGATIF pada KEEMPAT sel. Di rezim 6x, mengikuti
rekomendasi agen MERUGIKAN pengguna -- sementara pada greedy tetap +110,7 mnt.
Ini kebalikan penuh dari temuan di rezim 4x (sel4 +54,3 mnt, tertinggi dari
seluruh lengan). Batas keberlakuan artefak, dan wajib dilaporkan.


## E.5 — Pemahaman preferensi lintas kondisi

Menguji butir E6 pada seluruh 8 kondisi, bukan hanya baku. Ambang kebetulan
terkalibrasi lewat greedy (agen buta-pengguna): `pref_dalam` = k/6 = **0,500**,
`pref_primer` = 1/6 = **0,167**, diverifikasi lewat uji k=2 vs k=3 (meleset <1,5 poin
persen).

Uji yang menentukan bukan "apakah sel4 di atas ambang", melainkan **apakah Modul P
menaikkannya** — yaitu `efek_P = (sel3+sel4)/2 − (sel1+sel2)/2`. Pembanding paling
tajam adalah **sel1**, yang gerbang Modul P-nya nol alias modulnya mati total.

In [16]:
for m, ambang in [('pref_dalam', 0.5), ('pref_primer', 1/6)]:
    baris = []
    for nm, basis, k in KOND:
        r = {'kondisi': nm}
        for sel in SUF:
            r[sel] = nilai(tag(basis, sel, k), m)[0] - ambang
        baris.append(r)
    print()
    print(f'=== {m} — SELISIH dari ambang kebetulan {ambang:.4f} ===')
    display(pd.DataFrame(baris).set_index('kondisi').round(4))


=== pref_dalam — SELISIH dari ambang kebetulan 0.5000 ===


,sel1,sel2,sel3,sel4
kondisi,,,,
baku,-0.0097,0.0163,0.0182,0.0125
"trust 0,3",0.0186,0.0111,0.0090,0.0102
"trust 0,7",0.0363,0.0444,0.0224,0.0294
"gamma 0,95",0.0203,0.0130,0.0308,0.0144
"gamma 0,999",0.0711,0.0203,0.0257,0.0292
"gw x0,5",0.0277,0.0140,0.0186,0.0113
gw x2,0.1101,0.0025,0.0125,0.0361
beban 6x,-0.0077,0.0646,0.0846,0.0155



=== pref_primer — SELISIH dari ambang kebetulan 0.1667 ===


,sel1,sel2,sel3,sel4
kondisi,,,,
baku,-0.0049,0.0131,0.0107,0.0031
"trust 0,3",0.0154,0.0012,0.0024,0.0010
"trust 0,7",0.0168,0.0170,0.0121,0.0115
"gamma 0,95",0.0127,0.0067,0.0191,0.0067
"gamma 0,999",0.0339,0.0087,0.0047,0.0204
"gw x0,5",0.0161,0.0078,0.0098,-0.0000
gw x2,0.0368,-0.0053,0.0048,0.0200
beban 6x,-0.0045,0.0298,0.0301,0.0110


In [17]:
baris = []
for nm, basis, k in KOND:
    d = {s: nilai(tag(basis, s, k), 'pref_dalam')[0] for s in SUF}
    baris.append({'kondisi': nm,
                  'efek_P': (d['sel3'] + d['sel4']) / 2 - (d['sel1'] + d['sel2']) / 2,
                  'efek_attn': (d['sel2'] + d['sel4']) / 2 - (d['sel1'] + d['sel3']) / 2,
                  'sel1 (P MATI)': d['sel1'], 'sel4 (P aktif)': d['sel4'],
                  'sel4 > sel1': d['sel4'] > d['sel1']})
ep = pd.DataFrame(baris).set_index('kondisi')
display(ep.round(4))
print(f"efek_P rerata = {ep['efek_P'].mean():+.4f}   "
      f"positif di {(ep['efek_P']>0).sum()}/{len(ep)} kondisi")
print(f"sel4 > sel1 hanya di {ep['sel4 > sel1'].sum()}/{len(ep)} kondisi")
print()
print('>> Modul P MENURUNKAN ketepatan menebak preferensi. Lengan yang modulnya MATI')
print('   (sel1) justru lebih sering menebak benar. Ini bukti lebih kuat daripada')
print('   "setara kebetulan": arah efeknya negatif dan konsisten.')

,efek_P,efek_attn,sel1 (P MATI),sel4 (P aktif),sel4 > sel1
kondisi,,,,,
baku,0.0121,0.0101,0.4903,0.5125,True
"trust 0,3",-0.0053,-0.0032,0.5186,0.5102,False
"trust 0,7",-0.0144,0.0076,0.5363,0.5294,False
"gamma 0,95",0.0060,-0.0118,0.5203,0.5144,False
"gamma 0,999",-0.0183,-0.0236,0.5711,0.5292,False
"gw x0,5",-0.0059,-0.0105,0.5277,0.5113,False
gw x2,-0.0320,-0.0420,0.6101,0.5361,False
beban 6x,0.0216,0.0016,0.4923,0.5155,True


efek_P rerata = -0.0045   positif di 3/8 kondisi
sel4 > sel1 hanya di 2/8 kondisi

>> Modul P MENURUNKAN ketepatan menebak preferensi. Lengan yang modulnya MATI
   (sel1) justru lebih sering menebak benar. Ini bukti lebih kuat daripada
   "setara kebetulan": arah efeknya negatif dan konsisten.


In [18]:
# Uji penjelasan tandingan: apakah pref_dalam tinggi sekadar mencerminkan
# "sering menyarankan stasiun populer" (SPKLU_00 pop 21,9, jauh di atas lainnya)?
from scipy.stats import spearmanr
xs, ys, zs = [], [], []
for nm, basis, k in KOND:
    for s in SUF:
        t = tag(basis, s, k)
        xs.append(nilai(t, 'pref_dalam')[0])
        ys.append(nilai(t, 'rec_entropy')[0])
        zs.append(nilai(t, 'entropi_spklu_pengguna')[0])
for lbl, arr in [('rec_entropy', ys), ('entropi_spklu_pengguna', zs)]:
    r, p = spearmanr(xs, arr)
    print(f'pref_dalam vs {lbl:24s} rho={r:+.3f} (p={p:.3f}, n={len(xs)})')
print()
print('Keduanya LEMAH & tak signifikan -> konsentrasi rekomendasi BUKAN penjelasannya.')
print('pref_dalam tampak mengukur hal yang memang dimaksudkan.')

pref_dalam vs rec_entropy              rho=-0.168 (p=0.358, n=32)
pref_dalam vs entropi_spklu_pengguna   rho=+0.172 (p=0.348, n=32)

Keduanya LEMAH & tak signifikan -> konsentrasi rekomendasi BUKAN penjelasannya.
pref_dalam tampak mengukur hal yang memang dimaksudkan.


In [19]:
# Dari mana keunggulan acc sel4 berasal, bila bukan dari menebak preferensi?
baris = []
for nm, basis, k in KOND:
    for s in ['sel1', 'sel4']:
        t = tag(basis, s, k)
        baris.append({'kondisi': nm, 'sel': s,
                      'acc': nilai(t, 'acc')[0],
                      'pref_dalam': nilai(t, 'pref_dalam')[0],
                      'patuh|pref ADA': nilai(t, 'patuh_bila_pref_ada')[0],
                      'patuh|pref TIADA': nilai(t, 'patuh_bila_pref_tiada')[0]})
src = pd.DataFrame(baris).pivot(index='kondisi', columns='sel')
display(src.round(4))
d14 = (src[('patuh|pref TIADA', 'sel4')] - src[('patuh|pref TIADA', 'sel1')])
d_ada = (src[('patuh|pref ADA', 'sel4')] - src[('patuh|pref ADA', 'sel1')])
print()
print(f"selisih sel4-sel1 pada 'patuh bila pref TIADA' : rerata {d14.mean():+.4f}"
      f"  positif di {(d14>0).sum()}/{len(d14)} kondisi")
print(f"selisih sel4-sel1 pada 'patuh bila pref ADA'   : rerata {d_ada.mean():+.4f}"
      f"  positif di {(d_ada>0).sum()}/{len(d_ada)} kondisi")
print()
print('>> Keunggulan sel4 terpusat pada kemampuan MENGGESER pengguna ketika stasiun')
print('   favoritnya TIDAK ditawarkan -- yaitu DAYA PERSUASI, bukan ketepatan menebak.')
print('   Untuk pemerataan, itu justru kemampuan yang dibutuhkan: agen yang pandai')
print('   menebak favorit akan MENGUKUHKAN ketimpangan, bukan memperbaikinya.')

acc         pref_dalam         patuh|pref ADA         patuh|pref TIADA        
sel            sel1    sel4       sel1    sel4           sel1    sel4             sel1    sel4
kondisi                                                                                       
baku         0.6700  0.7194     0.4903  0.5125         0.9703  0.9675           0.3820  0.4580
beban 6x     0.6564  0.6829     0.4923  0.5155         0.9694  0.9667           0.3526  0.3829
gamma 0,95   0.7173  0.7155     0.5203  0.5144         0.9677  0.9674           0.4452  0.4481
gamma 0,999  0.7378  0.7121     0.5711  0.5292         0.9706  0.9697           0.4219  0.4221
gw x0,5      0.7258  0.7175     0.5277  0.5113         0.9677  0.9695           0.4548  0.4535
gw x2        0.7458  0.7326     0.6101  0.5361         0.9732  0.9671           0.3752  0.4605
trust 0,3    0.6722  0.6674     0.5186  0.5102         0.9575  0.9569           0.3635  0.3656
trust 0,7    0.7525  0.7643     0.5363  0.5294         0.9800  0.9797           0.4869  0.5191


selisih sel4-sel1 pada 'patuh bila pref TIADA' : rerata +0.0285  positif di 7/8 kondisi
selisih sel4-sel1 pada 'patuh bila pref ADA'   : rerata -0.0015  positif di 1/8 kondisi

>> Keunggulan sel4 terpusat pada kemampuan MENGGESER pengguna ketika stasiun
   favoritnya TIDAK ditawarkan -- yaitu DAYA PERSUASI, bukan ketepatan menebak.
   Untuk pemerataan, itu justru kemampuan yang dibutuhkan: agen yang pandai
   menebak favorit akan MENGUKUHKAN ketimpangan, bukan memperbaikinya.


## E.6 — Ringkasan Blok E

In [20]:
print('RINGKASAN BLOK E'.center(76, '='))
print()
print(f"sel4 peringkat-1 pada {(pk == 1).sum().sum()}/{pk.size} sel-metrik lintas 8 kondisi")
print(f"interaksi gini_UTIL menguntungkan pada {n_baik}/{len(g)} kondisi")
print()
print('Batas klaim yang WAJIB dinyatakan:')
print('  - Kondisi lingkungan (trust, beban, gw) menguji validitas eksternal;')
print('    kondisi gamma menguji sensitivitas penyetelan. JANGAN digabung.')
print('  - Efek gamma teredam `max_step_gap=4` -> hasil null wajar, bukan bukti')
print('    bahwa gamma tak penting.')
print('  - Dgn n=3 checkpoint per sel-kondisi, satu checkpoint yang kolaps sudah')
print('    cukup menggeser rerata. Selalu baca kolom rentang di E.3.')

==============================RINGKASAN BLOK E==============================

sel4 peringkat-1 pada 24/48 sel-metrik lintas 8 kondisi
interaksi gini_UTIL menguntungkan pada 5/3 kondisi

Batas klaim yang WAJIB dinyatakan:
  - Kondisi lingkungan (trust, beban, gw) menguji validitas eksternal;
    kondisi gamma menguji sensitivitas penyetelan. JANGAN digabung.
  - Efek gamma teredam `max_step_gap=4` -> hasil null wajar, bukan bukti
    bahwa gamma tak penting.
  - Dgn n=3 checkpoint per sel-kondisi, satu checkpoint yang kolaps sudah
    cukup menggeser rerata. Selalu baca kolom rentang di E.3.
